# IT4653 runner (Kaggle)
Notebook chỉ bootstrap và gọi CLI; không đặt training logic tại đây. Bật GPU trong Notebook settings trước khi chạy.

In [ ]:
import platform

import torch
import torchvision

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), "Hãy bật GPU trong Kaggle Settings rồi restart session"

Clone repository của nhóm vào `/kaggle/working`, rồi thay đường dẫn bên dưới. Chỉ official-run khi commit và versions khớp protocol.

In [ ]:
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<owner>/<repo>.git"
COMMIT_SHA = "<commit-sha-from-freeze-record>"
PROJECT_DIR = Path("/kaggle/working/BTL_IT4653")
assert "<owner>" not in REPO_URL and not COMMIT_SHA.startswith("<"), "Điền URL và commit đã freeze"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "--all", "--tags"], check=True)
subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", COMMIT_SHA], check=True)
actual_sha = subprocess.check_output(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "HEAD"], text=True
).strip()
assert actual_sha == COMMIT_SHA, (actual_sha, COMMIT_SHA)
print("Pinned commit:", actual_sha)

In [ ]:
%cd /kaggle/working/BTL_IT4653
!python -m pip install -q -e . --no-deps
!python scripts/kaggle_check.py
!python scripts/check_configs.py
!python scripts/smoke_test.py
# Chỉ cần ở reproducibility gate ngày đầu; dùng CIFAR-10 đã Add Input.
# !python scripts/overfit_128.py

In [ ]:
# Chọn matrix và MỘT chunk. Mỗi --only chạy cả seed 42 và 2026.
MATRIX = "configs/matrices/member1_optimizer_norm.yaml"
ONLY_ID = "opt_sgd"
!python scripts/run_matrix.py {MATRIX} --dry-run
# Pilot khi matrix còn approved:false:
# !python scripts/run_matrix.py {MATRIX} --only {ONLY_ID} --allow-draft
# Sau khi matrix approved và protocol freeze, bỏ dấu # ở đúng một dòng.
# !python scripts/run_matrix.py {MATRIX} --only {ONLY_ID}
# Nếu session trước hỏng, KHÔNG xóa log; retry bằng attempt mới:
# !python scripts/run_matrix.py {MATRIX} --only {ONLY_ID} --retry-failed --attempt retry1

Cuối **mỗi** session phải export artifact trước khi Kaggle thu hồi máy. Save Notebook Version hoặc tải cả `.tar.gz` và `.sha256`. Khi quay lại, attach archive cũ rồi gọi `import_logs.py` trước khi chạy chunk tiếp. Checkpoint chỉ export sau khi đã freeze final selection.

In [ ]:
MEMBER = "member1"
PART = "part1"
ARCHIVE = f"/kaggle/working/{MEMBER}_logs_{PART}.tar.gz"
!python scripts/export_logs.py --output {ARCHIVE}
print("Download/Save Version:", ARCHIVE, "và", ARCHIVE + ".sha256")
# Session mới, sau khi attach archive cũ:
# !python scripts/import_logs.py /kaggle/input/<dataset-name>/<archive-name>.tar.gz

Người 2 là integrator: import archive của cả ba thành viên, rồi chạy `aggregate_results.py --strict` và `plot_results.py`. Shared anchor nằm trong archive thành viên 1. Quy trình đầy đủ, gồm export hai checkpoint final, nằm tại `docs/KAGGLE_WORKFLOW.md`.